# GTFS — Aggregate processed outputs into a final GeoPackage (v2)

Walks `output/` for every `routes_<date>.gpkg` + matching `.arrivals.csv` pair
produced by `gtfs_analysis_v2.ipynb` (Phase C) and aggregates them into a
single consolidated GeoPackage of stop-level delay metrics.

## What's new in v2

In addition to the v1 pipeline:

* **`route_type_name`** (and the underlying numeric `route_type`) is looked up
  from the `routes` layer of each daily GPKG and merged onto every output row,
  so each is tagged with the GTFS modal type ("Bus", "Tram", etc.).
* **`stop_frac`** (the snapped fractional position of a stop along its route
  line, taken from the daily `stops` layer) is carried onto every stop-metrics
  row.
* **A new `routes` layer** is added to the consolidated GPKG. One row per
  `(route_id, direction_id)` carries the route's `LineString` geometry plus a
  JSON-encoded `stop_sequence` of stop_ids ordered by `stop_frac`. A sidecar
  `routes_v2.geojson` is also emitted for direct use in web visualisations,
  so downstream code does not need to reconstruct route shape or stop order
  itself.
* `direction_id` is already a grouping key in v1 — preserved as-is so a
  bidirectional route still appears as **two rows** per stop (one per
  direction), each with its own delay distribution.

The aggregated GPKG is written to a **separate file** (`stop_metrics_v2.gpkg`)
so the v1 output is preserved.

## What it does

1. **Auto-detect** all date-pair outputs in `output/` and prints an inventory.
2. **Build the study-area stop set** — uses the polygon saved at
   `data_checkpoint/study_area_polygon.gpkg` to filter to stops inside the
   study area (Greater Manchester for this project). Falls back to LSOA + LAD
   lookup if the checkpoint is missing.
3. **Build route + stop lookups** from each daily GPKG:
   * `route_id → (route_type, route_type_name)` — for per-arrival tagging.
   * `(route_id, direction_id) → (LineString, geometry_method, …)` — for the
     consolidated `routes` layer.
   * `(route_id, direction_id, stop_id) → stop_frac` — for stop ordering and
     per-row positional tagging.
4. **Concatenate all arrivals**, attach `date` / `day_type` / `time_period` /
   `route_type` / `route_type_name`, drop rows without a measured arrival.
5. **Aggregate** per `(route_id, direction_id, stop_id)` into a wide table
   with overall + per-time-period blocks of stats. Each row also carries
   `route_type`, `route_type_name`, and `stop_frac`.
6. **Write one GeoPackage** with five layers:
   `stop_metrics_weekday`, `stop_metrics_weekend`, `stop_metrics_all`,
   `routes` (LineStrings + ordered `stop_sequence`), and `meta`.
7. **Top-delayed-routes** report by `(day_type × time_period)` and a
   per-`route_type_name` headline summary.

## Time bands (UK transport peaks)

| Band              | Hours       |
|-------------------|-------------|
| `morning_offpeak` | 00:00–06:59 |
| `morning_peak`    | 07:00–08:59 |
| `midday_offpeak`  | 09:00–15:59 |
| `evening_peak`    | 16:00–18:59 |
| `evening_offpeak` | 19:00–23:59 |


## 1. Configuration

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import pandas as pd
import geopandas as gpd

# ── Paths ───────────────────────────────────────────────────────────────────
PROJ_DIR        = Path(r"D:\2026_03-Bus_Project")
OUTPUT_DIR      = PROJ_DIR / "output"
CHECKPOINT_DIR  = PROJ_DIR / "data_checkpoint"
DATA_OTHER_DIR  = PROJ_DIR / "data_other"            # LSOA + LU fallback

# Aggregated output goes here — one GPKG with multiple layers + sidecar CSVs.
# v2 writes to a separate file so the v1 output is preserved.
AGG_OUT_DIR     = OUTPUT_DIR / "aggregate"
AGG_OUT_DIR.mkdir(parents=True, exist_ok=True)
AGG_GPKG        = AGG_OUT_DIR / "stop_metrics_v2.gpkg"

# ── Time bands ──────────────────────────────────────────────────────────────
BAND_BINS   = [0, 7, 9, 16, 19, 24]
BAND_LABELS = [
    "morning_offpeak", "morning_peak",
    "midday_offpeak",  "evening_peak", "evening_offpeak",
]

# ── Filtering knobs ─────────────────────────────────────────────────────────
# Which `arrival_source` tags count as a real arrival measurement.
# Phase C v2 uses `extrapolated_pre` / `extrapolated_post` for the
# pre-service-fix arrivals — we keep them as valid by default.
VALID_ARRIVAL_SOURCES = {
    "interpolated", "dwell",
    "extrapolated_pre", "extrapolated_post",
}

# On-time threshold (minutes — `|delay| <= ON_TIME_TOLERANCE` is "on time")
ON_TIME_TOLERANCE = 1.0

# LSOA + LAD fallback (used only if checkpoint polygon is missing)
GM_LADS = ["Wigan", "Bolton", "Bury", "Rochdale", "Oldham",
           "Tameside", "Stockport", "Salford", "Trafford", "Manchester"]


## 2. Auto-detect processed outputs

Pairs each `routes_<date>.gpkg` with its matching `.arrivals.csv`. Files that
are missing one or the other are listed but skipped. A summary table reports
file size / arrival-row count so you can spot anomalies before processing.


In [2]:
import re
from datetime import datetime as _dt

DATE_RE = re.compile(r"routes_(\d{8})(?:\.arrivals\.csv|\.gpkg)$")


def _is_versioned(p: Path) -> bool:
    """Skip backup/versioned files like `routes_20250618_v2.gpkg`."""
    return "_v" in p.stem


def discover_outputs(output_dir: Path):
    """Return a DataFrame of detected output pairs, indexed by date."""
    found = {}
    for f in output_dir.iterdir():
        if not f.is_file() or _is_versioned(f):
            continue
        m = DATE_RE.search(f.name)
        if not m:
            continue
        date_str = m.group(1)
        slot = found.setdefault(date_str, {"gpkg": None, "csv": None,
                                            "gpkg_size": 0, "csv_size": 0})
        if f.suffix.lower() == ".csv":
            slot["csv"] = f
            slot["csv_size"] = f.stat().st_size
        elif f.suffix.lower() == ".gpkg":
            slot["gpkg"] = f
            slot["gpkg_size"] = f.stat().st_size

    rows = []
    for date_str, slot in sorted(found.items()):
        try:
            d = _dt.strptime(date_str, "%Y%m%d").date()
            day_name = d.strftime("%a")
            day_type = "Weekend" if d.weekday() >= 5 else "Weekday"
        except ValueError:
            day_name = day_type = "?"
        rows.append({
            "date":     date_str,
            "weekday":  day_name,
            "day_type": day_type,
            "gpkg":     slot["gpkg"].name if slot["gpkg"] else "",
            "csv":      slot["csv"].name  if slot["csv"]  else "",
            "gpkg_MB":  round(slot["gpkg_size"] / 1_048_576, 1),
            "csv_MB":   round(slot["csv_size"] / 1_048_576, 1),
            "complete": bool(slot["gpkg"] and slot["csv"]),
        })
    return pd.DataFrame(rows), found


inventory, _output_index = discover_outputs(OUTPUT_DIR)
print(f"Found {len(inventory)} dated outputs in {OUTPUT_DIR}")
print(f"  complete pairs:   {int(inventory['complete'].sum())}")
print(f"  missing GPKG:     {(inventory['gpkg'] == '').sum()}")
print(f"  missing CSV:      {(inventory['csv'] == '').sum()}")
inventory


Found 10 dated outputs in D:\2026_03-Bus_Project\output
  complete pairs:   10
  missing GPKG:     0
  missing CSV:      0


,date,weekday,day_type,gpkg,csv,gpkg_MB,csv_MB,complete
0,20250618,Wed,Weekday,routes_20250618.gpkg,routes_20250618.arrivals.csv,342.9,56.9,True
1,20250619,Thu,Weekday,routes_20250619.gpkg,routes_20250619.arrivals.csv,226.3,10.2,True
2,20250620,Fri,Weekday,routes_20250620.gpkg,routes_20250620.arrivals.csv,881.9,137.9,True
3,20250621,Sat,Weekend,routes_20250621.gpkg,routes_20250621.arrivals.csv,813.8,127.4,True
4,20250622,Sun,Weekend,routes_20250622.gpkg,routes_20250622.arrivals.csv,627.5,76.0,True
5,20250623,Mon,Weekday,routes_20250623.gpkg,routes_20250623.arrivals.csv,761.7,141.4,True
6,20250624,Tue,Weekday,routes_20250624.gpkg,routes_20250624.arrivals.csv,877.1,131.7,True
7,20250625,Wed,Weekday,routes_20250625.gpkg,routes_20250625.arrivals.csv,877.9,131.0,True
8,20250626,Thu,Weekday,routes_20250626.gpkg,routes_20250626.arrivals.csv,877.3,136.6,True
9,20250627,Fri,Weekday,routes_20250627.gpkg,routes_20250627.arrivals.csv,873.8,136.4,True


## 3. Study-area stops + route metadata, geometry, and stop ordering

Two passes over the daily GPKGs:

1. **Stops** — Loads `data_checkpoint/study_area_polygon.gpkg` (the dissolved
   GM polygon from Phase A; LSOA + LAD fallback). For every daily GPKG we
   read its `stops` layer and union the stop_ids that lie inside the polygon.
   The same pass also collects `(route_id, direction_id, stop_id) → stop_frac`
   so we know each stop's fractional position along its route line.
2. **Routes** — Reads each daily GPKG's `routes` layer (which carries
   `route_type`, `route_type_name`, and the route's `LineString` geometry).
   Builds two lookups:
   * `route_id → (route_type, route_type_name)` for the per-arrival tagging
     (most-recent non-null value wins).
   * `(route_id, direction_id) → (LineString, geometry_method, …)` for the
     consolidated `routes` layer (most-recent non-null geometry wins).


In [3]:
import time as _t


def load_study_area_polygon():
    """Prefer the Phase A checkpoint; fall back to LSOA + LAD union."""
    ckpt = CHECKPOINT_DIR / "study_area_polygon.gpkg"
    if ckpt.exists():
        print(f"  using checkpoint polygon: {ckpt.relative_to(PROJ_DIR)}")
        gdf = gpd.read_file(ckpt)
        return gdf.to_crs("EPSG:4326")

    print(f"  checkpoint not found — rebuilding from LSOA + LAD lookup")
    lu_path   = DATA_OTHER_DIR / "PCD_OA21_LSOA21_MSOA21_LAD_AUG23_UK_LU.csv"
    lsoa_path = DATA_OTHER_DIR / "LSOA2021_EW_BFC_V10.gpkg"
    if not (lu_path.exists() and lsoa_path.exists()):
        raise FileNotFoundError(
            "Need either the checkpoint polygon at "
            f"{ckpt.relative_to(PROJ_DIR)} or the LSOA + LU files at "
            f"{DATA_OTHER_DIR.relative_to(PROJ_DIR)}/")
    lu = (pd.read_csv(lu_path, encoding="cp1252", low_memory=False,
                      usecols=["lsoa21cd", "ladnm"])
            .rename(columns={"lsoa21cd": "LSOA21CD"})
            .drop_duplicates("LSOA21CD"))
    lsoa = gpd.read_file(lsoa_path)[["LSOA21CD", "geometry"]]
    return (lsoa.merge(lu[lu["ladnm"].isin(GM_LADS)], on="LSOA21CD")
                 .dissolve()[["geometry"]]
                 .to_crs("EPSG:4326"))


print("[1/3] Loading study-area polygon ...")
study_area = load_study_area_polygon()
print(f"      polygon: {study_area.geom_type.iloc[0]}, "
      f"{study_area.to_crs('EPSG:27700').area.iloc[0] / 1e6:.1f} km²")

# ── Stops inside the polygon (union across dates) + stop_frac lookup ───────
print(f"\n[2/3] Reading stops + stop_frac from "
      f"{int(inventory['complete'].sum())} GPKGs ...")
study_area_stop_ids = set()
stop_attrs = {}        # stop_id → {geometry, stop_name}
stop_frac_rows = []    # collected across dates; reduced to most-recent below

for date_str, slot in _output_index.items():
    if slot["gpkg"] is None:
        continue
    t0 = _t.monotonic()
    stops = gpd.read_file(slot["gpkg"], layer="stops")
    if stops.crs != study_area.crs:
        stops = stops.to_crs(study_area.crs)
    inside = gpd.clip(stops, study_area)
    n_in   = len(inside)
    study_area_stop_ids.update(inside["stop_id"].tolist())
    # Vectorised attr capture (avoids the slow per-row iterrows loop)
    new_ids = set(inside["stop_id"]) - set(stop_attrs)
    if new_ids:
        first_seen = inside[inside["stop_id"].isin(new_ids)].drop_duplicates("stop_id")
        for _, r in first_seen.iterrows():
            stop_attrs[r["stop_id"]] = {"geometry":  r["geometry"],
                                        "stop_name": r.get("stop_name", "")}

    # Collect stop_frac per (route_id, direction_id, stop_id) for this date.
    if {"route_id", "direction_id", "stop_id", "stop_frac"}.issubset(stops.columns):
        sub = stops[["route_id", "direction_id", "stop_id", "stop_frac"]].copy()
        sub["date"] = date_str
        stop_frac_rows.append(sub)

    print(f"  {date_str}: {len(stops):>5,} total stops → "
          f"{n_in:>5,} inside study area  ({_t.monotonic()-t0:4.1f}s)")

print(f"\n  Union → {len(study_area_stop_ids):,} unique stops in the study area")

stops_gdf = gpd.GeoDataFrame(
    [{"stop_id":   sid,
      "stop_name": v["stop_name"],
      "geometry":  v["geometry"]} for sid, v in stop_attrs.items()],
    crs=study_area.crs,
)

# Most-recent non-null stop_frac wins per (route_id, direction_id, stop_id)
if not stop_frac_rows:
    raise RuntimeError("No stop_frac data found in any daily GPKG.")
stop_frac_all = pd.concat(stop_frac_rows, ignore_index=True)
stop_frac_all["route_id"]     = stop_frac_all["route_id"].astype(str)
stop_frac_all["direction_id"] = stop_frac_all["direction_id"].astype(str)
stop_frac_all["stop_id"]      = stop_frac_all["stop_id"].astype(str)
stop_frac_all = stop_frac_all.sort_values(
    ["route_id", "direction_id", "stop_id", "date"])
stop_frac = (stop_frac_all
             .dropna(subset=["stop_frac"])
             .drop_duplicates(["route_id", "direction_id", "stop_id"], keep="last")
             [["route_id", "direction_id", "stop_id", "stop_frac"]]
             .reset_index(drop=True))
print(f"  stop_frac entries: {len(stop_frac):,} unique "
      f"(route_id × direction_id × stop_id)")

# ── Route metadata + geometry lookup ───────────────────────────────────────
print(f"\n[3/3] Building route metadata + geometry lookup ...")
route_meta_rows = []   # for route_id → (route_type, route_type_name)
route_geom_rows = []   # for (route_id, direction_id) → LineString + provenance
GEOM_COLS = ["route_id", "direction_id", "route_short_name",
             "route_type", "route_type_name",
             "geometry_method", "n_stops_covered", "n_trips_observed",
             "geometry"]

for date_str, slot in _output_index.items():
    if slot["gpkg"] is None:
        continue
    routes = gpd.read_file(slot["gpkg"], layer="routes")

    # route_id-level metadata
    cols_meta = [c for c in ["route_id", "route_type", "route_type_name"]
                 if c in routes.columns]
    if "route_id" in cols_meta:
        route_meta_rows.append(
            routes[cols_meta].drop_duplicates("route_id").assign(date=date_str))

    # (route_id, direction_id) geometry + provenance
    cols_geom = [c for c in GEOM_COLS if c in routes.columns]
    if {"route_id", "direction_id", "geometry"}.issubset(cols_geom):
        sub = routes[cols_geom].copy()
        sub["date"] = date_str
        route_geom_rows.append(sub)

if not route_meta_rows:
    raise RuntimeError("No `routes` layers found in any GPKG.")

# route_id → (route_type, route_type_name) — most-recent non-null wins
route_meta_all = pd.concat(route_meta_rows, ignore_index=True)
route_meta_all = route_meta_all.sort_values(["route_id", "date"])
route_meta = (route_meta_all
              .dropna(subset=["route_type_name"])
              .drop_duplicates("route_id", keep="last")
              [["route_id", "route_type", "route_type_name"]]
              .reset_index(drop=True))
route_meta["route_id"] = route_meta["route_id"].astype(str)
print(f"  {len(route_meta):,} unique route_ids resolved")
print("  route_type_name distribution:")
print(route_meta["route_type_name"].value_counts().rename_axis("route_type_name")
      .to_string())

# (route_id, direction_id) → LineString — most-recent non-null wins
if not route_geom_rows:
    raise RuntimeError("No route geometries found in any GPKG.")
route_geom_all = pd.concat(route_geom_rows, ignore_index=True)
route_geom_all["route_id"]     = route_geom_all["route_id"].astype(str)
route_geom_all["direction_id"] = route_geom_all["direction_id"].astype(str)
route_geom_all = route_geom_all.sort_values(["route_id", "direction_id", "date"])
route_geom = (route_geom_all
              .dropna(subset=["geometry"])
              .drop_duplicates(["route_id", "direction_id"], keep="last")
              .drop(columns="date")
              .reset_index(drop=True))
route_geom = gpd.GeoDataFrame(route_geom, geometry="geometry", crs="EPSG:4326")
print(f"  {len(route_geom):,} unique (route_id × direction_id) geometries")


[1/3] Loading study-area polygon ...
  using checkpoint polygon: data_checkpoint\study_area_polygon.gpkg
      polygon: Polygon, 1276.0 km²

[2/3] Reading stops + stop_frac from 10 GPKGs ...
  20250618: 53,827 total stops → 49,182 inside study area  ( 1.2s)
  20250619: 53,827 total stops → 49,182 inside study area  ( 0.6s)
  20250620: 53,827 total stops → 49,182 inside study area  ( 0.6s)
  20250621: 53,827 total stops → 49,182 inside study area  ( 0.6s)
  20250622: 53,827 total stops → 49,182 inside study area  ( 0.6s)
  20250623: 53,827 total stops → 49,182 inside study area  ( 0.6s)
  20250624: 53,827 total stops → 49,182 inside study area  ( 0.6s)
  20250625: 53,827 total stops → 49,182 inside study area  ( 0.7s)
  20250626: 53,827 total stops → 49,182 inside study area  ( 0.6s)
  20250627: 53,827 total stops → 49,182 inside study area  ( 0.6s)

  Union → 12,556 unique stops in the study area
  stop_frac entries: 53,827 unique (route_id × direction_id × stop_id)

[3/3] Building rou

## 4. Load and classify arrivals

Concatenates every `arrivals.csv` (only the dates with a complete pair),
filters to the study-area stop set, attaches `date` / `day_type` /
`time_period`, **merges in `route_type` / `route_type_name`** from the
lookup built above, and drops rows without a measured arrival
(`arrival_source ∉ VALID_ARRIVAL_SOURCES`).


In [4]:
dfs = []
total_rows_seen = 0
total_rows_kept = 0

valid_pairs = inventory[inventory["complete"]].copy()
print(f"Loading {len(valid_pairs)} arrivals CSVs ...")

for _, row in valid_pairs.iterrows():
    csv_path = _output_index[row["date"]]["csv"]
    t0 = _t.monotonic()
    # Force route_id and direction_id to string up front to silence
    # `DtypeWarning: Columns (1,3) have mixed types`.
    df = pd.read_csv(csv_path, dtype={"route_id": str, "direction_id": str})
    total_rows_seen += len(df)

    # If the CSV came from a pre-direction-fix run, fill in direction_id=0
    if "direction_id" not in df.columns:
        df["direction_id"] = "0"
    df["direction_id"] = df["direction_id"].astype(str)

    # Filter: stops in study area + valid arrival source
    df = df[df["stop_id"].isin(study_area_stop_ids)]
    if "arrival_source" in df.columns:
        df = df[df["arrival_source"].isin(VALID_ARRIVAL_SOURCES)]
    # Defensive — drop NaN delay rows
    df = df.dropna(subset=["delay_min"])

    df["date"] = pd.to_datetime(row["date"], format="%Y%m%d")
    total_rows_kept += len(df)
    dfs.append(df)
    print(f"  {row['date']} ({row['weekday']}, {row['day_type']:<7s}): "
          f"{len(df):>7,} kept of total CSV rows "
          f"({_t.monotonic()-t0:4.1f}s)")

if not dfs:
    raise RuntimeError("No arrivals to aggregate — inventory was empty?")

arr = pd.concat(dfs, ignore_index=True)
del dfs
print(f"\nTotal arrivals concatenated: {len(arr):,} "
      f"({100 * total_rows_kept / total_rows_seen:.1f}% of {total_rows_seen:,} CSV rows)")

# ── Merge route_type / route_type_name ─────────────────────────────────────
arr = arr.merge(route_meta, on="route_id", how="left")
n_missing_rt = arr["route_type_name"].isna().sum()
print(f"\nrows with no route_type_name lookup: {n_missing_rt:,} "
      f"({100 * n_missing_rt / len(arr):.2f}%)")
arr["route_type_name"] = arr["route_type_name"].fillna("Unknown")

# Day type + time period
arr["day_type"] = arr["date"].dt.dayofweek.apply(
    lambda d: "Weekend" if d >= 5 else "Weekday")
arr["hour"] = arr["scheduled_arrival"].str.split(":").str[0].astype(int) % 24
arr["time_period"] = pd.cut(arr["hour"], bins=BAND_BINS,
                             labels=BAND_LABELS, right=False, include_lowest=True)

# Sanity: per-date count + day_type
print("\nDate × day_type distribution:")
print(arr.groupby(["date", "day_type"]).size().rename("n_arrivals").to_string())
print("\nrow count by route_type_name:")
print(arr["route_type_name"].value_counts().to_string())


Loading 10 arrivals CSVs ...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_29420\2782567301.py:13: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, dtype={"route_id": str, "direction_id": str})


  20250618 (Wed, Weekday): 261,774 kept of total CSV rows ( 0.8s)
  20250619 (Thu, Weekday):   1,512 kept of total CSV rows ( 0.1s)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_29420\2782567301.py:13: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, dtype={"route_id": str, "direction_id": str})
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_29420\2782567301.py:13: DtypeWarning: Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, dtype={"route_id": str, "direction_id": str})


  20250620 (Fri, Weekday): 727,210 kept of total CSV rows ( 2.0s)


...

  20250625 (Wed, Weekday): 574,315 kept of total CSV rows ( 2.0s)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_29420\2782567301.py:13: DtypeWarning: Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, dtype={"route_id": str, "direction_id": str})


  20250626 (Thu, Weekday): 728,668 kept of total CSV rows ( 2.0s)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_29420\2782567301.py:13: DtypeWarning: Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, dtype={"route_id": str, "direction_id": str})


  20250627 (Fri, Weekday): 713,726 kept of total CSV rows ( 2.0s)

Total arrivals concatenated: 5,333,705 (76.4% of 6,983,651 CSV rows)

rows with no route_type_name lookup: 0 (0.00%)

Date × day_type distribution:
date        day_type
2025-06-18  Weekday     261774
2025-06-19  Weekday       1512
2025-06-20  Weekday     727210
2025-06-21  Weekend     627641
2025-06-22  Weekend     365576
2025-06-23  Weekday     727331
2025-06-24  Weekday     605952
2025-06-25  Weekday     574315
2025-06-26  Weekday     728668
2025-06-27  Weekday     713726

row count by route_type_name:
route_type_name
Bus    5333705


## 5. Headline metrics — `day_type × time_period` (and per `route_type_name`)

In [5]:
def pct_on_time(s):  return float((s.abs() <= ON_TIME_TOLERANCE).mean() * 100)
def pct_late(s):     return float((s >  ON_TIME_TOLERANCE).mean() * 100)

summary = (arr.groupby(["day_type", "time_period"], observed=True)["delay_min"]
              .agg(n_arrivals="count",
                   avg_delay_min="mean",
                   median_delay_min="median",
                   pct_on_time=pct_on_time,
                   pct_late=pct_late)
              .round(2))


def pivot(col, decimals=2):
    return (summary[col].unstack("day_type")
            .round(decimals)
            .rename_axis(None, axis=1))

print("=== Average delay (minutes) ===")
display(pivot("avg_delay_min"))
print("=== Median delay (minutes) ===")
display(pivot("median_delay_min"))
print(f"=== % within {ON_TIME_TOLERANCE} min (on-time) ===")
display(pivot("pct_on_time", decimals=1))
print(f"=== % more than {ON_TIME_TOLERANCE} min late ===")
display(pivot("pct_late", decimals=1))

# Per-mode headline (uses median, which is robust to extreme delay outliers)
print("\n=== Per route_type_name (median delay min, % on-time) ===")
mode_summary = (arr.groupby("route_type_name", observed=True)["delay_min"]
                  .agg(n_arrivals="count",
                       median_delay_min="median",
                       pct_on_time=pct_on_time,
                       pct_late=pct_late)
                  .round(2)
                  .sort_values("n_arrivals", ascending=False))
display(mode_summary)


=== Average delay (minutes) ===


,Weekday,Weekend
time_period,,
morning_offpeak,94.86,15.84
morning_peak,10.07,4.18
midday_offpeak,11.90,5.17
evening_peak,52.68,4.89
evening_offpeak,206.97,10.38


=== Median delay (minutes) ===


,Weekday,Weekend
time_period,,
morning_offpeak,1.10,1.40
morning_peak,0.98,1.07
midday_offpeak,1.96,1.72
evening_peak,2.07,1.32
evening_offpeak,2.41,1.93


=== % within 1.0 min (on-time) ===


,Weekday,Weekend
time_period,,
morning_offpeak,36.6,32.3
morning_peak,31.8,36.3
midday_offpeak,25.2,27.3
evening_peak,23.5,30.2
evening_offpeak,23.4,26.8


=== % more than 1.0 min late ===


,Weekday,Weekend
time_period,,
morning_offpeak,52.2,57.6
morning_peak,49.5,51.3
midday_offpeak,63.4,61.1
evening_peak,62.2,55.3
evening_offpeak,68.4,64.0



=== Per route_type_name (median delay min, % on-time) ===


,n_arrivals,median_delay_min,pct_on_time,pct_late
route_type_name,,,,
Bus,5333705,1.75,26.74,61.03


## 6. Per-route × direction × stop wide table

One row per `(route_id, direction_id, stop_id)`. Each row is tagged with
`route_short_name`, `route_type`, `route_type_name`, `stop_name`,
`stop_frac` (fractional position along the route line), and coordinates.
Then come blocks of metrics — overall (`_day`) and one block per time period:

| Metric                | Description                                  |
|-----------------------|----------------------------------------------|
| `n_arrivals_<suffix>` | Count of valid arrivals                      |
| `min/Q1/Q2/Q3/max`    | Delay quantiles (minutes)                    |
| `pct_on_time`         | % within ±`ON_TIME_TOLERANCE` min            |
| `pct_late`            | % more than `ON_TIME_TOLERANCE` min late     |
| `list_delay`          | Stringified list of every delay value, e.g. `"[2.1, -0.5, 3.0]"` |


In [6]:
def agg_metrics(s, suffix):
    """Return a dict of metrics for one delay Series, suffixed for column naming."""
    vals = s.dropna()
    if len(vals) == 0:
        return {f"n_arrivals_{suffix}":   0,
                f"Q1_{suffix}":           None,
                f"Q2_{suffix}":           None,
                f"Q3_{suffix}":           None,
                f"min_{suffix}":          None,
                f"max_{suffix}":          None,
                f"pct_on_time_{suffix}":  None,
                f"pct_late_{suffix}":     None,
                f"list_delay_{suffix}":   "[]"}
    return {f"n_arrivals_{suffix}":  int(len(vals)),
            f"Q1_{suffix}":          round(float(vals.quantile(0.25)), 2),
            f"Q2_{suffix}":          round(float(vals.quantile(0.50)), 2),
            f"Q3_{suffix}":          round(float(vals.quantile(0.75)), 2),
            f"min_{suffix}":         round(float(vals.min()), 2),
            f"max_{suffix}":         round(float(vals.max()), 2),
            f"pct_on_time_{suffix}": round(float((vals.abs() <= ON_TIME_TOLERANCE).mean() * 100), 2),
            f"pct_late_{suffix}":    round(float((vals >  ON_TIME_TOLERANCE).mean() * 100), 2),
            f"list_delay_{suffix}":  str(vals.round(2).tolist())}


GROUP_KEYS = ["route_id", "direction_id", "stop_id"]
# Identifier columns kept on every output row. v2 adds `route_type` /
# `route_type_name` and `stop_frac` to retain modal type and the stop's
# fractional position along the route line.
ID_COLS    = ["route_id", "route_short_name",
              "route_type", "route_type_name",
              "direction_id",
              "stop_id", "stop_name", "stop_frac"]


def build_wide_table(df_subset, label: str):
    """Aggregate df_subset by GROUP_KEYS into a wide GeoDataFrame."""
    print(f"  building wide table for '{label}' "
          f"({len(df_subset):,} arrivals) ...", flush=True)
    t0 = _t.monotonic()
    rows = []
    for keys, grp in df_subset.groupby(GROUP_KEYS, observed=True):
        row = dict(zip(GROUP_KEYS, keys))
        # Carry along the per-route / per-stop scalar identifiers.
        row["route_short_name"] = grp["route_short_name"].iloc[0]
        row["route_type"]       = grp["route_type"].iloc[0] \
                                    if "route_type" in grp.columns else None
        row["route_type_name"]  = grp["route_type_name"].iloc[0] \
                                    if "route_type_name" in grp.columns else None
        row.update(agg_metrics(grp["delay_min"], "day"))
        for period in BAND_LABELS:
            sub = grp.loc[grp["time_period"] == period, "delay_min"]
            row.update(agg_metrics(sub, period))
        rows.append(row)
    wide = pd.DataFrame(rows)

    # Attach stop_frac per (route_id, direction_id, stop_id).
    wide = wide.merge(stop_frac, on=GROUP_KEYS, how="left")

    # join geometry + stop_name
    gdf = stops_gdf.merge(wide, on="stop_id", how="inner")

    # add coords (WGS84 + BNG)
    gdf["lon"] = gdf.geometry.x.round(6)
    gdf["lat"] = gdf.geometry.y.round(6)
    bng        = gdf.to_crs("EPSG:27700")
    gdf["easting"]  = bng.geometry.x.round(1)
    gdf["northing"] = bng.geometry.y.round(1)

    # reorder
    coord_cols      = ["lon", "lat", "easting", "northing"]
    day_cols        = [c for c in gdf.columns if c.endswith("_day")]
    period_cols     = [c for c in gdf.columns
                       if any(c.endswith(f"_{p}") for p in BAND_LABELS)]
    gdf = gdf[ID_COLS + coord_cols + day_cols + period_cols + ["geometry"]]
    print(f"    {len(gdf):,} rows  ({_t.monotonic()-t0:4.1f}s)")
    return gdf


# Build all three layers
print("Aggregating per-route × direction × stop ...")
wide_all      = build_wide_table(arr,                                 "all")
wide_weekday  = build_wide_table(arr[arr["day_type"] == "Weekday"],   "weekday")
wide_weekend  = build_wide_table(arr[arr["day_type"] == "Weekend"],   "weekend")


Aggregating per-route × direction × stop ...
  building wide table for 'all' (5,333,705 arrivals) ...
    45,050 rows  (259.2s)
  building wide table for 'weekday' (4,340,488 arrivals) ...
    44,935 rows  (250.2s)
  building wide table for 'weekend' (993,217 arrivals) ...
    22,513 rows  (155.7s)


## 7. Write the consolidated GeoPackage

One GPKG with five layers:

* `stop_metrics_all` / `stop_metrics_weekday` / `stop_metrics_weekend` —
  the per-`(route_id, direction_id, stop_id)` wide tables.
* `routes` — one row per `(route_id, direction_id)` carrying the route's
  `LineString` geometry, `geometry_method`, and a JSON-encoded
  `stop_sequence` listing the stop_ids in `stop_frac` order (filtered to
  study-area stops). A sidecar `routes_v2.geojson` is also written for
  direct use in web visualisations.
* `meta` — records exactly which dates and source files were aggregated.

Sidecar CSVs (without geometry) are written next to the GPKG for the
stop-metrics layers.

v2 writes to `stop_metrics_v2.gpkg` so the v1 output is preserved.


In [7]:
import json

# Wipe the GPKG so layers don't pile up across re-runs
if AGG_GPKG.exists():
    AGG_GPKG.unlink()

LAYERS = {
    "stop_metrics_all":     wide_all,
    "stop_metrics_weekday": wide_weekday,
    "stop_metrics_weekend": wide_weekend,
}

print(f"Writing → {AGG_GPKG}")
first = True
for name, gdf in LAYERS.items():
    gdf.to_file(AGG_GPKG, layer=name, driver="GPKG",
                mode="w" if first else "a")
    csv_path = AGG_OUT_DIR / f"{name}_v2.csv"
    gdf.drop(columns="geometry").to_csv(csv_path, index=False)
    print(f"  {name:<22s} {len(gdf):>6,} rows  →  layer + {csv_path.name}")
    first = False

# ── Routes layer ────────────────────────────────────────────────────────────
# One row per (route_id, direction_id): LineString + ordered stop_sequence
# (sorted by stop_frac, filtered to study-area stops).
ordered = (stop_frac[stop_frac["stop_id"].isin(study_area_stop_ids)]
           .sort_values(["route_id", "direction_id", "stop_frac"])
           .groupby(["route_id", "direction_id"])
           .agg(stop_sequence=("stop_id", list),
                n_stops_in_area=("stop_id", "count"))
           .reset_index())

route_layer = route_geom.merge(ordered, on=["route_id", "direction_id"],
                               how="inner")
route_layer["stop_sequence"] = route_layer["stop_sequence"].apply(json.dumps)

route_id_cols = [c for c in [
    "route_id", "route_short_name",
    "route_type", "route_type_name",
    "direction_id",
    "geometry_method",
    "n_stops_covered", "n_trips_observed",
    "n_stops_in_area", "stop_sequence",
] if c in route_layer.columns]
route_layer = gpd.GeoDataFrame(
    route_layer[route_id_cols + ["geometry"]],
    geometry="geometry", crs="EPSG:4326")

route_layer.to_file(AGG_GPKG, layer="routes", driver="GPKG", mode="a")
geojson_path = AGG_OUT_DIR / "routes_v2.geojson"
if geojson_path.exists():
    geojson_path.unlink()
route_layer.to_file(geojson_path, driver="GeoJSON")
print(f"  {'routes':<22s} {len(route_layer):>6,} rows  →  layer + {geojson_path.name}")

# ── Metadata layer ─────────────────────────────────────────────────────────
# Records the exact inputs used for traceability
meta = inventory[inventory["complete"]].assign(
    on_time_tolerance_min = ON_TIME_TOLERANCE,
    valid_sources         = ", ".join(sorted(VALID_ARRIVAL_SOURCES)),
    n_unique_stops        = len(study_area_stop_ids),
    n_unique_routes       = len(route_meta),
    n_route_geometries    = len(route_layer),
    n_arrivals_total      = len(arr),
)
# GPKG can't take an attribute-only table without geometry, so we attach a
# representative point (study-area centroid) to every row.
centroid = study_area.geometry.iloc[0].centroid
meta_gdf = gpd.GeoDataFrame(meta, geometry=[centroid] * len(meta),
                             crs=study_area.crs)
meta_gdf.to_file(AGG_GPKG, layer="meta", driver="GPKG", mode="a")
meta.to_csv(AGG_OUT_DIR / "meta_v2.csv", index=False)
print(f"  {'meta':<22s} {len(meta):>6,} rows  →  layer + meta_v2.csv")
print(f"\nDone. All outputs in {AGG_OUT_DIR}")


Writing → D:\2026_03-Bus_Project\output\aggregate\stop_metrics_v2.gpkg
  stop_metrics_all       45,050 rows  →  layer + stop_metrics_all_v2.csv
  stop_metrics_weekday   44,935 rows  →  layer + stop_metrics_weekday_v2.csv
  stop_metrics_weekend   22,513 rows  →  layer + stop_metrics_weekend_v2.csv
  routes                  1,305 rows  →  layer + routes_v2.geojson
  meta                       10 rows  →  layer + meta_v2.csv

Done. All outputs in D:\2026_03-Bus_Project\output\aggregate


## 8. Top-delayed routes (diagnostics)

Quick top-5 most-delayed routes per `day_type × time_period`, with
`route_type_name` shown alongside. Useful as a sanity check against the
GeoPackage you just wrote.


In [8]:
top_delayed = (
    arr.groupby(["day_type", "time_period",
                 "route_short_name", "route_type_name"], observed=True)["delay_min"]
       .agg(n_arrivals="count", avg_delay_min="mean")
       .round(2)
       .reset_index()
       .sort_values(["day_type", "time_period", "avg_delay_min"],
                    ascending=[True, True, False]))

top5 = top_delayed.groupby(["day_type", "time_period"], observed=True).head(5)
top5


,day_type,time_period,route_short_name,route_type_name,n_arrivals,avg_delay_min
68,Weekday,morning_offpeak,475,Bus,13,665.55
203,Weekday,morning_offpeak,37,Bus,238,657.28
81,Weekday,morning_offpeak,527,Bus,143,654.08
73,Weekday,morning_offpeak,511,Bus,324,547.10
199,Weekday,morning_offpeak,358,Bus,490,438.83
754,Weekday,morning_peak,767,Bus,153,480.91
914,Weekday,morning_peak,954,Bus,250,406.20
852,Weekday,morning_peak,880,Bus,183,370.50
377,Weekday,morning_peak,450,Bus,71,325.26
806,Weekday,morning_peak,826,Bus,109,304.65
